In [ ]:
from typing import *

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

from core.feature_type import FeatureType
from feature_writer.pump_feature_writer import REGRESSOR_OFFSETS
from feature_writer.utils import create_dataset

# Load data from pumps/cross_sections
df: pd.DataFrame = create_dataset()
df["is_pumped"] = df["currency_pair"] == df["pumped_currency_pair"]  # create bool column is_pumped

In [ ]:
df.shape

In [ ]:
# Overall we have data for 465 pumps
df["pump_hash"].nunique()

In [ ]:
# cross-section statistics
(
    df.groupby("pump_hash")
    .agg(
        pump_time=("pump_time", "first"),
        cross_section_size=("pump_time", "count")
    )
    .sort_values(by="pump_time", ascending=True)
    .plot(x="pump_time", y="cross_section_size", title="Cross-section size over time")
)
plt.show()

<h4>Feature distributions and cross-sectional standardisation</h4>

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)

<p>Now we will go through each FeatureType and display distribution stats and fix gaps in data</p>

<h4>Powerlaw features</h4>

In [ ]:
powerlaw_cols: List[str] = FeatureType.POWERLAW_ALPHA.col_names(offsets=REGRESSOR_OFFSETS)
df[powerlaw_cols].describe().T

In [ ]:
# clip pawerlaw features to range [1, 2]
df[powerlaw_cols] = df[powerlaw_cols].clip(1, 2)
df[powerlaw_cols].describe().T

In [ ]:
df[powerlaw_cols].isna().sum()

In [ ]:
df[powerlaw_cols] = df[powerlaw_cols].fillna(-1)

<h4>Asset return features</h4>

<p><b>Note: returns are measured in pips</b></p>
<p>Here we observe that anticipated behaviour as time window grows the min/max and std become higher in absolute terms</p>

In [ ]:
asset_return_cols: List[str] = FeatureType.ASSET_RETURN.col_names(offsets=REGRESSOR_OFFSETS)
df[asset_return_cols].describe().T

In [ ]:
df[asset_return_cols].isna().sum()

<h4>Asset return zscores</h4>

In [ ]:
asset_return_zscore_cols: List[str] = FeatureType.ASSET_RETURN_ZSCORE.col_names(offsets=REGRESSOR_OFFSETS)
df[asset_return_zscore_cols].describe().T

In [ ]:
df[asset_return_zscore_cols].isna().sum()

<h4>Quote abs zscore</h4>

In [ ]:
quote_abs_zscore_cols: List[str] = FeatureType.QUOTE_ABS_ZSCORE.col_names(offsets=REGRESSOR_OFFSETS)
df[quote_abs_zscore_cols].describe().T

<h4>Share of long trades</h4>

In [ ]:
share_of_long_trades_cols: List[str] = FeatureType.SHARE_OF_LONG_TRADES.col_names(offsets=REGRESSOR_OFFSETS)
df[share_of_long_trades_cols].describe().T

<h4>Slippage imbalance</h4>

In [ ]:
slippage_imbalance_cols: List[str] = FeatureType.SLIPPAGE_IMBALANCE.col_names(offsets=REGRESSOR_OFFSETS)
df[slippage_imbalance_cols].describe().T

In [ ]:
df[slippage_imbalance_cols].isna().sum()

<h4>Flow imbalance</h4>

In [ ]:
flow_imbalance_cols: List[str] = FeatureType.FLOW_IMBALANCE.col_names(offsets=REGRESSOR_OFFSETS)
df[flow_imbalance_cols].describe().T

<h4>Num trades</h4>

In [ ]:
num_trades_cols: List[str] = FeatureType.NUM_TRADES.col_names(offsets=REGRESSOR_OFFSETS)
df[num_trades_cols].describe().T

In [ ]:
# overall NaNs
df[df.isna().any(axis=1)].shape

<h4>Plot feature distributions before and after cross-sectional standardisation</h4>

$$X_{\text{std}} = \frac{X - \bar{X}}{\sigma_{X}}$$

In [ ]:
# plot feature distributions before cross-sectional standardisation
cols_to_scale: List[str] = (
        asset_return_cols + asset_return_zscore_cols + quote_abs_zscore_cols + powerlaw_cols
)

dfs: List[pd.DataFrame] = []

for pump_hash, df_cross_section in tqdm(df.groupby("pump_hash")):
    df_cross_section = df_cross_section.reset_index(drop=True)
    for col in cols_to_scale:
        df_cross_section[col] = (df_cross_section[col] - df_cross_section[col].mean()) / df_cross_section[col].std()
    dfs.append(df_cross_section)

df_scaled: pd.DataFrame = pd.concat(dfs)
df_scaled = df_scaled.reset_index(drop=True)
df_scaled.head(2)

In [ ]:
from core.time_utils import NamedTimeDelta
import seaborn as sns

fig, axs = plt.subplots(len(FeatureType), 2, figsize=(20, 50))
use_offset: NamedTimeDelta = NamedTimeDelta.ONE_DAY

# create two smaller dataframes with less cross-sections to plot
selected_pump_hashes: np.array = np.random.choice(df["pump_hash"].unique(), 5,
                                                  replace=False)  # use 5 pumps for plotting
df_small: pd.DataFrame = df[df["pump_hash"].isin(selected_pump_hashes)]
df_scaled_small: pd.DataFrame = df_scaled[df_scaled["pump_hash"].isin(selected_pump_hashes)]

feature: FeatureType

for (ax1, ax2), feature in tqdm(zip(axs, list(FeatureType))):
    col_name: str = feature.col_name(offset=use_offset)
    # Plot pumps without standardisation
    sns.histplot(
        data=df_small,
        x=col_name,
        hue="pump_hash",
        ax=ax1, legend=False, alpha=0.05, bins=50, kde=True, stat="probability"
    )
    ax1.set_title(f"No standardisation: {col_name}")
    for pump_hash in selected_pump_hashes:
        ax1.axvline(
            x=df_small.loc[df_small["is_pumped"] & (df_small["pump_hash"] == pump_hash), col_name].iloc[0],
            color="red", linestyle="--"
        )

    sns.histplot(
        data=df_scaled_small,
        x=col_name,
        hue="pump_hash",
        ax=ax2, legend=False, alpha=0.05, bins=50, kde=True, stat="probability"
    )
    ax2.set_title(f"Cross-section standardisation: {col_name}")
    for pump_hash in selected_pump_hashes:
        ax2.axvline(
            x=df_scaled_small.loc[
                df_scaled_small["is_pumped"] & (df_scaled_small["pump_hash"] == pump_hash), col_name].iloc[0],
            color="red", linestyle="--"
        )

<h4>Split data and train the first model</h4>

In [ ]:
# Train/val/test split
df_train, df_val, df_test = (
    df_scaled[df_scaled["pump_time"] <= "2020-09-01"].copy(),  # train sample
    df_scaled[(df_scaled["pump_time"] > "2020-09-01") & (df_scaled["pump_time"] <= "2021-05-01")].copy(),  # val sample
    df_scaled[df_scaled["pump_time"] > "2021-05-01"].copy(),  # test sample
)

# Sample statistics
pumps = np.array([df_train["is_pumped"].sum(), df_val["is_pumped"].sum(), df_test["is_pumped"].sum()])
overall_observations = np.array([df_train.shape[0], df_val.shape[0], df_test.shape[0]])

avg_cross = np.array([
    df_train.groupby("pump_hash")["currency_pair"].count().mean(),
    df_val.groupby("pump_hash")["currency_pair"].count().mean(),
    df_test.groupby("pump_hash")["currency_pair"].count().mean()
])

df_cross_stats = pd.DataFrame({
    "Positive": pumps,
    "Negative": overall_observations - pumps,
    "Total": overall_observations,
    "Average Crosssection Size": avg_cross
}).T.round(1)

df_cross_stats.columns = ["Train", "Validation", "Test"]
df_cross_stats

In [ ]:
from sklearn.metrics import precision_recall_curve, PrecisionRecallDisplay


def plot_precision_recall(df: pd.DataFrame, model_probas: Dict[str, np.array],
                          figsize: Tuple[int, int] = (10, 10)) -> None:
    """Plot Precision Recall curve for the model"""
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111)

    for model_name, y_proba in model_probas.items():
        precision, recall, _ = precision_recall_curve(y_true=df["is_pumped"], y_score=y_proba)
        PrecisionRecallDisplay(precision=precision, recall=recall).plot(
            ax=ax, label=f"PRAUC: {model_name} - {auc(recall, precision):.4f}"
        )

    f_scores = np.linspace(0.1, 0.8, num=10)

    for f_score in f_scores:
        x = np.linspace(0.01, 1)
        y = f_score * x / (2 * x - f_score)
        (l,) = ax.plot(x[y >= 0], y[y >= 0], color="blue", alpha=0.2)
        ax.annotate("f1={0:0.1f}".format(f_score), xy=(0.9, y[45] + 0.02))

    plt.legend(loc="upper right")
    plt.title("Precision Recall curves")

<h4>Baseline Logistic Regression</h4>

In [ ]:
mean_crosssection_size: int = df.groupby("pump_hash")["currency_pair"].count().mean()
print(f"Mean crosssection size across the whole dataset is {mean_crosssection_size:.3f}")

In [ ]:
# define regression columns and target
target: str = "is_pumped"
reg_cols: List[str] = (
        asset_return_cols + asset_return_zscore_cols +
        quote_abs_zscore_cols + share_of_long_trades_cols +
        powerlaw_cols + slippage_imbalance_cols +
        flow_imbalance_cols + num_trades_cols
)

<h4>Baseline Logistic Regression</h4>

In [ ]:
from sklearn.linear_model import LogisticRegression

# Train baseline Logistic Regression
model_lr = LogisticRegression(
    max_iter=int(1e10),
    class_weight={0: 1, 1: mean_crosssection_size},
)

# model_lr.fit(df_train[reg_cols], df_train[target])

# probas_pred_lr_val = model_lr.predict_proba(df_val[reg_cols])[:, 1]

<h4>RandomForestClassifier</h4>

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    criterion="gini",
    n_estimators=1000,
    max_depth=5,
    n_jobs=-1,
    max_samples=.8
)

model_rf.fit(X=df_train[reg_cols], y=df_train[target])

In [ ]:
probas_pred_rf: np.ndarray = model_rf.predict_proba(df_test[reg_cols])[:, 1]

plot_precision_recall(df=df_test, model_probas={"RandomForestClassifier": probas_pred_rf})

<h4>TOP-K or Hit ratio</h4>

<p>This metric measures the change of getting the only positive given we take K highest logits</p>

In [ ]:
def calculate_topk(df: pd.DataFrame, probas_pred: np.ndarray, topk_bins: List[int]) -> Dict[int, float]:
    _df = df.copy()
    _df["probas_pred"] = probas_pred
    count_by_bins: Dict[int, int] = {}

    for pump_hash, df_cross_section in _df.groupby("pump_hash"):
        df_cross_section = df_cross_section.sort_values(by="probas_pred", ascending=False)
        for K in topk_bins:
            contains_pump: bool = df_cross_section.iloc[:K]["is_pumped"].any()
            count_by_bins[K] = count_by_bins.get(K, 0) + contains_pump

    n_sections: int = df["pump_hash"].nunique()
    return {K: count / n_sections for K, count in count_by_bins.items()}

<h4>TOP-K%</h4>

In [ ]:
def calculate_topk_percent(df: pd.DataFrame, probas_pred: np.ndarray, bins: List[float]) -> Dict[float, float]:
    _df = df.copy()
    _df["probas_pred"] = probas_pred
    count_by_bins: Dict[float, int] = {}

    for pump_hash, df_cross_section in _df.groupby("pump_hash"):
        df_cross_section = df_cross_section.sort_values(by="probas_pred", ascending=False)
        n_rows = len(df_cross_section)

        for pct_bin in bins:
            k = max(1, int(np.ceil(n_rows * pct_bin)))
            contains_pump: bool = df_cross_section.iloc[:k]["is_pumped"].any()
            count_by_bins[pct_bin] = count_by_bins.get(pct_bin, 0) + contains_pump

    n_sections: int = df["pump_hash"].nunique()
    return {pct_bin: count / n_sections for pct_bin, count in count_by_bins.items()}

<h4>TOP-K% AUC</h4>

In [ ]:
from sklearn.metrics import auc


def calculate_topk_percent_auc(df: pd.DataFrame, probas_pred: np.ndarray) -> float:
    topks: Dict[float, float] = calculate_topk_percent(df=df, probas_pred=probas_pred, bins=np.arange(0, 1.01, 0.005))
    return auc(x=list(topks.keys()), y=list(topks.values()))

<b>See if all works as expected</b>

In [ ]:
probas_pred_rf: np.ndarray = model_rf.predict_proba(X=df_test[reg_cols])[:, 1]

calculate_topk(df=df_test, probas_pred=probas_pred_rf, topk_bins=[1, 2, 5, 10, 20, 30])

In [ ]:
calculate_topk_percent(df=df_test, probas_pred=probas_pred_rf, bins=[0, 0.01, 0.05, 0.1, 0.2, 0.5])

In [ ]:
calculate_topk_percent_auc(df=df_test, probas_pred=probas_pred_rf)

In [ ]:
pd.DataFrame({
    "feature": model_rf.feature_names_in_,
    "feature_importance": model_rf.feature_importances_
}).sort_values(by="feature_importance", ascending=False).head(20)

<h4>CatboostClassifier</h4>

In [ ]:
from catboost import CatBoostClassifier, Pool
from functools import partial
from optuna.trial import Trial

import optuna


def objective_catboost(
        trial: Trial, df_train: pd.DataFrame, df_val: pd.DataFrame, reg_cols: List[str], target: str
) -> float:
    cb_params: Dict[str, Any] = {
        "objective": "Logloss",
        "num_boost_round": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.7, 1),
        "subsample": trial.suggest_float("subsample", 0.7, 1),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "auto_class_weights": "Balanced",
    }

    ptrain: Pool = Pool(data=df_train[reg_cols], label=df_train[target])
    pval: Pool = Pool(data=df_val[reg_cols], label=df_val[target])

    model: CatBoostClassifier = CatBoostClassifier(**cb_params, verbose=False)
    model.fit(ptrain, eval_set=pval, early_stopping_rounds=50, use_best_model=True)

    probas: np.array = model.predict_proba(pval)[:, 1]
    topk_percent_auc: float = calculate_topk_percent_auc(df=df_val, probas_pred=probas)

    return topk_percent_auc

In [ ]:
study_catboost = optuna.create_study(direction="maximize")

study_catboost.optimize(
    partial(
        objective_catboost, df_train=df_train, df_val=df_val, reg_cols=reg_cols, target="is_pumped"
    ),
    n_trials=100
)

In [ ]:
cb_params = {
    "objective": "Logloss",
    "sampling_frequency": "PerTree",
    "num_boost_round": 1000,
}

cb_params.update(study_catboost.best_params)

In [ ]:
ptrain: Pool = Pool(data=df_train[reg_cols], label=df_train["is_pumped"])
pval: Pool = Pool(data=df_val[reg_cols], label=df_val["is_pumped"])
ptest: Pool = Pool(data=df_test[reg_cols], label=df_test["is_pumped"])

model_cb = CatBoostClassifier(**cb_params, verbose=False, use_best_model=True)
model_cb.fit(ptrain, eval_set=pval, early_stopping_rounds=50, plot=True)

In [ ]:
probas_pred_cb: np.ndarray = model_cb.predict_proba(df_test[reg_cols])[:, 1]
probas_pred_rf: np.ndarray = model_rf.predict_proba(df_test[reg_cols])[:, 1]

In [ ]:
calculate_topk_percent(df=df_test, probas_pred=probas_pred_cb, bins=[0, 0.01, 0.05, 0.1, 0.2, 0.5])

In [ ]:
calculate_topk(df=df_test, probas_pred=probas_pred_cb, topk_bins=[1, 2, 5, 10, 20, 30])

In [ ]:
calculate_topk_percent_auc(df=df_test, probas_pred=probas_pred_rf)

In [ ]:
plot_precision_recall(
    df=df_test, model_probas={
        "RandomForestClassifier": probas_pred_rf,
        "CatboostClassifier": probas_pred_cb
    }
)

In [ ]:
pd.DataFrame({
    "feature": model_cb.feature_names_,
    "feature_importance": model_cb.feature_importances_
}).sort_values(by="feature_importance", ascending=False).head(20)